# AssemLens task verification on ZGX Nano

This notebook is the verifier stage, separate from action recognition. It consumes the teammate's observed-action output and predicts `correct`, `mistake`, `correction`, or `uncertain`.

Run this notebook in the VS Code Remote SSH window connected to the Nano. Do not replace the vendor PyTorch/CUDA installation.

In [1]:
import os, platform, shutil, subprocess, sys
import torch
print(sys.executable)
print(platform.machine(), torch.__version__)
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('BF16:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
print('FFmpeg:', shutil.which('ffmpeg'))
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported()
assert shutil.which('ffmpeg')


/home/hp15/git_happens/assemlens/.venv/bin/python
aarch64 2.14.0+cu130
True NVIDIA GB10
BF16: True
FFmpeg: /usr/bin/ffmpeg


## 1. Inspect the prepared verification subset

The dataset has already been prepared on this Nano. This notebook intentionally does not rebuild or download data when you use Run All. The existing run contains 75 matched examples.

In [2]:
from pathlib import Path
import json, os, subprocess, sys
REPO_ROOT = Path('/home/hp15/git_happens/assemlens')
RUN_ROOT = REPO_ROOT / 'runs' / 'verification_100'
EXAMPLES = RUN_ROOT / 'examples.jsonl'
METADATA = RUN_ROOT / 'metadata.json'
assert RUN_ROOT.is_dir(), f'Missing run directory: {RUN_ROOT}'
assert EXAMPLES.is_file(), f'Missing examples: {EXAMPLES}'
assert METADATA.is_file(), f'Missing metadata: {METADATA}'
print(json.loads(METADATA.read_text()))


{'n': 75, 'requested': 100, 'label_counts': {'correct': 25, 'mistake': 25, 'correction': 25}, 'skipped': [], 'time_unit': 'frames', 'source': 'https://github.com/assembly-101/assembly101-mistake-detection'}


Inspect the label balance and a few examples before loading the VLM. The output includes the exact annotation file and timestamps used for every record.

In [3]:
rows = [json.loads(x) for x in EXAMPLES.read_text().splitlines() if x.strip()]
from collections import Counter
print(Counter(row['ground_truth_assessment'] for row in rows))
print(json.dumps(rows[0], indent=2)[:5000])


Counter({'correct': 25, 'mistake': 25, 'correction': 25})
{
  "example_id": "nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253:0:C10404_rgb",
  "sequence_id": "nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253",
  "video_id": "/home/hp15/.cache/huggingface/hub/datasets--cvml-nus--assembly101/blobs/267592b58435ffc3434e0f585241210aff7d717112066bc18aca56df965211f4",
  "view_id": "C10404_rgb",
  "expected_step": {
    "step_id": "nusar-2021_action_both_9011-b06b_9011_user_id_2021-02-01_154253:0",
    "instruction": "Attach interior with chassis",
    "verb": "attach",
    "objects": [
      "interior",
      "chassis"
    ],
    "source": "Assembly101 mistake annotation; ordering is sequence-contextual"
  },
  "previous_actions": [],
  "previous_verified_state": {
    "completed_steps": [],
    "unresolved_mistakes": [],
    "state_summary": "Derived benchmark context; not a physical-state tracker"
  },
  "before_frames": [
    "/home/hp15/git_happens/assemlens/

## 2. Run the zero-shot verification baseline

This is intentionally not fine-tuning. It establishes whether a VLM can use expected-step and temporal context before any verifier training.

In [4]:
baseline_path = RUN_ROOT / 'baseline.json'
run_env = os.environ.copy()
run_env['PYTHONPATH'] = str(REPO_ROOT) + os.pathsep + run_env.get('PYTHONPATH', '')
subprocess.run([sys.executable, str(REPO_ROOT/'scripts'/'run_verifier_baseline.py'), '--examples', str(EXAMPLES), '--output', str(baseline_path), '--limit', '75'], cwd=REPO_ROOT, env=run_env, check=True)
report = json.loads(baseline_path.read_text())
print(json.dumps(report['metrics'], indent=2))


W0923 21:26:20.533000 3279809 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0923 21:26:20.548000 3279809 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 113.20it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


1/75 uncertain gold=correct
2/75 uncertain gold=correct
3/75 uncertain gold=correct
4/75 uncertain gold=correct
5/75 uncertain gold=correct
6/75 uncertain gold=correct
7/75 uncertain gold=correct
8/75 uncertain gold=correct
9/75 uncertain gold=correct
10/75 uncertain gold=correct
11/75 uncertain gold=correct
12/75 uncertain gold=correct
13/75 uncertain gold=correct
14/75 uncertain gold=mistake
15/75 uncertain gold=mistake
16/75 uncertain gold=correct
17/75 uncertain gold=correct
18/75 uncertain gold=correct
19/75 uncertain gold=correct
20/75 uncertain gold=correct
21/75 uncertain gold=correct
22/75 uncertain gold=correct
23/75 uncertain gold=correct
24/75 uncertain gold=correct
25/75 SKIP missing frame: [Errno 2] No such file or directory: '/home/hp15/git_happens/assemlens/runs/verification_100/frames/00024/after/01.jpg'
26/75 uncertain gold=correct
27/75 uncertain gold=correct
28/75 uncertain gold=mistake
29/75 uncertain gold=mistake
30/75 uncertain gold=mistake
31/75 uncertain gold=c

## 3. Integration contract

Replace each record's `observed_action` with the teammate model's JSON (`action`, `verb`, `object`) before using the verifier in the live pipeline. Do not merge the action-recognition loss with the verifier loss until both stages have separate held-out evaluations.